# m01_compare_models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/javanmardi298/rag_ollama/blob/main/lessons/m01_compare_models/m01_compare_models.ipynb)

Run the cells in order. In each step: read, complete the TODOs in the code cell, run it, then run the check cell.

In [ ]:
# Setup (Colab: clones the repo; locally: skip if you are already inside it)
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', 'https://github.com/javanmardi298/rag_ollama.git', 'repo'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'repo/requirements.txt', '-e', 'repo'], check=True)
if os.path.exists('repo'): os.chdir('repo')
os.environ.setdefault('LEARN_PROVIDER', 'offline')   # offline | ollama | anthropic | opencode | openai_compatible


# گام ۱: تنظیم ارائه‌دهنده‌ی مدل از روی محیط (config)

## هدف این گام
تابعی می‌نویسی که متغیرهای محیطی (`LEARN_PROVIDER`، `LEARN_MODEL`، …) را می‌خواند، **اعتبارسنجی** می‌کند و یک تنظیم تمیز برمی‌گرداند. یادگیری اصلی: تنظیمات یک سیستم LLM را در **یک نقطه** جمع کن و خطای تنظیم را زود و واضح بگیر، نه وسط یک اجرای ۲۰ دقیقه‌ای.

## پیش‌نیاز
دیکشنری و استثنا در Python. [00_setup](#) را برای معنی هر ارائه‌دهنده ببین.

## مفهوم
ارائه‌دهنده‌ها فرق دارند ولی رابط ما یکی است (`chat` و `json`). تنها چیزی که عوض می‌شود **تنظیم** است:

| ارائه‌دهنده | `LEARN_MODEL` | شرط | پیش‌فرض مدل |
|---|---|---|---|
| `offline` | ندارد | هیچ | — |
| `ollama` | نام مدل | سرور Ollama | `qwen2.5:7b` |
| `anthropic` | نام مدل Claude | کلید `ANTHROPIC_API_KEY` | `claude-sonnet-5-5` |
| `opencode` | `provider/model` | OpenCode نصب باشد | ندارد (**الزامی**) |
| `openai_compatible` | نام مدل | `LEARN_API_BASE` | ندارد (**الزامی**) |

قواعد `resolve_config(env)`:
1. `provider = env.get("LEARN_PROVIDER", "offline")`؛ ناشناخته ← `ValueError("unknown provider ...")`.
2. `model = env.get("LEARN_MODEL")` یا پیش‌فرض جدول.
3. `opencode` بدون مدل، یا مدل بدون `/` ← `ValueError` (باید `provider/model` باشد).
4. `openai_compatible` بدون `LEARN_API_BASE` ← `ValueError`.
5. `anthropic` بدون کلید (`ANTHROPIC_API_KEY` در `env`) ← `ValueError` (اشتراک Claude.ai کلید نیست).
6. بودجه: `LEARN_BUDGET_USD` (پیش‌فرض ۰ = بدون سقف) به `float`؛ منفی یا غیرعددی ← `ValueError`.
7. خروجی: `{"provider", "model", "api_base", "budget_usd", "cloud": bool}` که `cloud` برای `anthropic` و `openai_compatible` درست است (داده به بیرون می‌رود).

## تمرین کد
فایل پاسخ: `work/m01_compare_models/step01_provider_config/answer.py`

## اجرا کن
```bash
python -m learn check m01_compare_models 1
```

## خروجی مورد انتظار
هفت ✓.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `float("abc")` استثنای دیگری داد | `ValueError` خودت را با پیام واضح بالا بیاور |
| مدل پیش‌فرض ollama خالی است | از جدول پیش‌فرض‌ها استفاده نکرده‌ای |

## دام‌ها
- هرگز **مقدار** کلید را چاپ یا لاگ نکن؛ فقط وجود آن را بررسی کن.
- پیش‌فرض‌های مدل با زمان کهنه می‌شوند؛ نام مدل‌های فعلی را از مستندات هر ارائه‌دهنده بگیر.

## گام بعد
[گام ۲: آزمون](#)


In [ ]:
"""Step 1: resolve the model provider configuration from environment variables."""
from __future__ import annotations

PROVIDERS = {"offline", "ollama", "anthropic", "opencode", "openai_compatible"}
DEFAULT_MODEL = {"offline": "", "ollama": "qwen2.5:7b", "anthropic": "claude-sonnet-5-5", "opencode": "", "openai_compatible": ""}


def resolve_config(env: dict) -> dict:
    """-> {"provider", "model", "api_base", "budget_usd", "cloud"}; raises ValueError with a clear message for bad settings."""
    # TODO 1: provider from LEARN_PROVIDER (default "offline"), unknown -> ValueError
    # TODO 2: model from LEARN_MODEL or DEFAULT_MODEL
    # TODO 3: opencode needs "provider/model"; openai_compatible needs LEARN_API_BASE; anthropic needs ANTHROPIC_API_KEY in env
    # TODO 4: budget_usd = float(LEARN_BUDGET_USD or 0); not a number or negative -> ValueError
    # TODO 5: cloud = provider in {"anthropic", "openai_compatible"}
    raise NotImplementedError("TODO 1-5: resolve_config")


In [ ]:
from learn.notebook import check_cell
check_cell('m01_compare_models', 'step01_provider_config', globals())

# گام ۲: آزمون قابل‌تکرار — دقت، تأخیر، خطا

## هدف این گام
تابع `run_benchmark` را می‌نویسی که چند مدل (هر کدام با رابط `chat`) را روی **همان سؤال‌ها** اجرا می‌کند و برای هر مدل دقت، میانگین تأخیر و تعداد خطاها را برمی‌گرداند.

## پیش‌نیاز
گام ۱ مفهوماً. این تابع به هیچ ارائه‌دهنده‌ی مشخصی وابسته نیست؛ هر شیئی با `.chat(messages).content` کافی است (در تست‌ها مدل ساختگی).

## مفهوم
یک آزمون خوب سه ویژگی دارد:
1. **یکسان:** همه‌ی مدل‌ها همان prompt ها را می‌بینند.
2. **خودکار:** درست/غلط را **کد** تعیین می‌کند، نه چشم تو: هر مورد یک تابع `check(answer) -> bool` دارد. (برای پرسش‌های باز، LLM-judge لازم است و آن را باید اعتبارسنجی کنی؛ اینجا پاسخ‌های کوتاه و قطعی می‌پرسیم.)
3. **مقاوم به شکست:** یک خطای شبکه یا زمان‌گذشت یک مدل نباید کل آزمون را بکشد؛ آن را **به‌عنوان خطا بشمار** و ادامه بده.

هر ردیف خروجی: `{"model", "accuracy", "mean_latency_s", "errors", "n"}`.
- `accuracy` = درست‌ها / `n` (خطا غلط حساب می‌شود).
- `mean_latency_s` = میانگین زمان فراخوانی‌های **موفق** (اگر هیچ موفقیتی نبود `None`).
- زمان را با `time.perf_counter()` بگیر.

## تمرین کد
فایل پاسخ: `work/m01_compare_models/step02_benchmark/answer.py`

`run_benchmark(models, cases, repeats=1)`:
- `models`: دیکشنری `نام ← شیء llm`.
- `cases`: لیست `{"prompt": str, "check": callable}`.
- هر مدل × هر مورد × `repeats` بار. پیام: `[{"role": "user", "content": prompt}]`.
- خروجی: لیست ردیف‌ها به ترتیب `models`.

## اجرا کن
```bash
python -m learn check m01_compare_models 2
python work/m01_compare_models/step02_benchmark/answer.py
```
اجرای دستی از متغیر `COMPARE` می‌خواند، مثلاً:
```powershell
$env:COMPARE="offline,ollama:qwen2.5:7b,anthropic:claude-sonnet-5-5,opencode:anthropic/claude-sonnet-5-5"
python work\m01_compare_models\step02_benchmark\answer.py
```
ارائه‌دهنده‌ای که در دسترس نیست رد می‌شود و دلیلش چاپ می‌شود.

## خروجی مورد انتظار
شش ✓. اجرای دستی با `offline` دقت پایین نشان می‌دهد (مدل ساختگی پاسخ‌ها را نمی‌داند): این درست است. با دو مدل واقعی، جدولی از دقت و تأخیر می‌بینی.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| یک خطا کل آزمون را می‌کشد | `try/except` دور هر فراخوانی نگذاشته‌ای |
| خطا به‌عنوان درست شمرده شد | در حالت خطا `ok=False` باید باشد |
| تأخیر خطاها هم در میانگین است | فقط موفق‌ها را بشمار |

## دام‌ها
- **ده سؤال یعنی اختلاف ۱۰٪ فقط یک سؤال است.** با آزمون کوچک نتیجه‌ی قطعی نگیر؛ مدل‌ها را روی ۳۰ تا ۱۰۰ مورد **از داده‌ی خودت** بسنج.
- temperature را صفر نگه دار و `repeats` را برای مدل‌های تصادفی بیشتر کن.
- اولین فراخوانی Ollama مدل را در حافظه بار می‌کند و کند است؛ یک فراخوانی گرم‌کردن (warm-up) را در تأخیر نشمار.
- تأخیر ابری شامل شبکه است؛ تأخیر محلی به سخت‌افزار تو بستگی دارد. مقایسه‌ی «منصفانه» نیست، فقط مقایسه‌ی «عملی» برای مورد استفاده‌ی توست.

## گام بعد
[گام ۳: انتخاب و گزارش](#)


In [ ]:
"""Step 2: the same cases on every model: accuracy, latency, errors."""
from __future__ import annotations

import os
import time

CASES = [
    {"prompt": "What is 17 + 25? Answer with the number only.", "check": lambda a: "42" in a},
    {"prompt": "What is the capital of France? One word.", "check": lambda a: "paris" in a.lower()},
    {"prompt": "Translate 'good morning' to German. Two words.", "check": lambda a: "guten morgen" in a.lower()},
    {"prompt": "Return only the word YES if 9 is greater than 3, otherwise NO.", "check": lambda a: a.strip().upper().startswith("YES")},
]


def run_benchmark(models: dict, cases: list[dict], repeats: int = 1) -> list[dict]:
    """models: name -> object with .chat(messages).content. Returns one row per model:
    {"model", "accuracy", "mean_latency_s", "errors", "n"}"""
    # TODO 1: for every model, case and repeat: time models[name].chat([{"role": "user", "content": prompt}]).content
    #         with time.perf_counter(); an exception counts as an error AND as a wrong answer (never stop the run);
    #         accuracy = correct / n; mean_latency_s = mean over the successful calls only (None if there were none)
    raise NotImplementedError("TODO 1: run_benchmark")


def build_models(spec: str) -> dict:
    """'offline,ollama:qwen2.5:7b,anthropic:claude-sonnet-5-5' -> {name: llm}; unavailable ones are skipped with a reason."""
    from learn import llm as learn_llm
    out = {}
    for item in filter(None, (x.strip() for x in spec.split(","))):
        provider, _, model = item.partition(":")
        try:
            out[item] = learn_llm.get(provider, model)
        except Exception as e:
            print(f"skipped {item}: {e}")
    return out


if __name__ == "__main__":
    models = build_models(os.environ.get("COMPARE", "offline"))
    for r in run_benchmark(models, CASES):
        print(r)


In [ ]:
from learn.notebook import check_cell
check_cell('m01_compare_models', 'step02_benchmark', globals())

# گام ۳: انتخاب مدل با قاعده و گزارش قابل‌انتشار

## هدف این گام
از ردیف‌های آزمون، با یک **قاعده‌ی از پیش تعیین‌شده** مدل را انتخاب می‌کنی (`choose`) و جدول markdown می‌سازی (`markdown_table`) که مستقیم در README پروژه یا رزومه‌ات می‌گذاری.

## پیش‌نیاز
خروجی گام ۲ (لیست ردیف‌ها). تست‌ها ردیف‌های آماده می‌دهند.

## مفهوم
«بهترین مدل» بدون **معیار** معنا ندارد. معیار را **قبل از دیدن نتیجه** بنویس:

> «از بین مدل‌هایی که دقتشان ≥ ۰٫۸ است و خطایی ندارند، کم‌تأخیرترین را بردار. اگر هیچ‌کدام نبود، هیچ.»

این قاعده سه ویژگی خوب دارد: ساده است، قابل‌تکرار است، و از «تنظیم معیار بعد از دیدن نتیجه» (که تقلب خودخواسته است) جلوگیری می‌کند. اگر هیچ مدلی نرسید، **جواب `None` است** و باید مدل بهتر بگیری یا کار را آسان‌تر کنی؛ نباید آستانه را پایین بیاوری تا یکی «برنده» شود.

`choose(rows, min_accuracy=0.8, max_errors=0)`:
- فقط ردیف‌هایی که `accuracy >= min_accuracy` و `errors <= max_errors` و `mean_latency_s` غیر `None` دارند.
- از بین آن‌ها کمترین `mean_latency_s`؛ اگر مساوی بود، بیشترین دقت؛ اگر باز مساوی بود، نام به ترتیب الفبا.
- خروجی: **نام** مدل یا `None`.

`markdown_table(rows)`:
```
| model | accuracy | latency (s) | errors |
|---|---|---|---|
| ollama:qwen2.5:7b | 0.75 | 1.20 | 0 |
```
- ردیف‌ها به ترتیب دقت نزولی (مساوی ← نام).
- دقت با ۲ رقم اعشار؛ تأخیر با ۲ رقم یا `-` اگر `None`.

## تمرین کد
فایل پاسخ: `work/m01_compare_models/step03_choose_and_report/answer.py`

## اجرا کن
```bash
python -m learn check m01_compare_models 3
python work/m01_compare_models/step03_choose_and_report/answer.py
```

## خروجی مورد انتظار
شش ✓ و چاپ یک جدول نمونه.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `TypeError` با `None` | مدلی بدون تأخیر (`None`) را قبل از مقایسه حذف نکرده‌ای |
| ترتیب جدول عوض می‌شود | مرتب‌سازی با کلید ترکیبی `(-accuracy, model)` |
| `-` چاپ نمی‌شود | حالت `None` را فرمت نکرده‌ای |

## دام‌ها
- هزینه‌ی ابری هم بخشی از انتخاب است: ستون «هزینه به‌ازای ۱۰۰۰ درخواست» را از قیمت واقعی خودت اضافه کن (تمرین).
- مدل برنده را در `config.yaml` پروژه ثبت کن و **تاریخ و نسخه‌ی** آزمون را کنارش بنویس؛ مدل‌ها و قیمت‌ها عوض می‌شوند.
- به نتیجه‌ی ۱۰ مورد تصمیم تجاری نگیر (گام ۲).

## جمع‌بندی m01 و قدم بعد
حالا می‌توانی بگویی «برای کار من مدل X را انتخاب کردم، چون روی ۵۰ نمونه‌ی خودم دقت … و تأخیر … داشت، با هزینه‌ی …». قدم بعد: همین آزمون را با داده‌ی خودت و `llm_judge` اعتبارسنجی‌شده برای خروجی‌های باز، و ثبت نتیجه در پنل (`results/`).


In [ ]:
"""Step 3: choose a model with a rule fixed in advance, and print a report table."""
from __future__ import annotations


def choose(rows: list[dict], min_accuracy: float = 0.8, max_errors: int = 0):
    """Among rows with accuracy >= min_accuracy, errors <= max_errors and a measured latency, return the NAME of the fastest
    (ties: higher accuracy, then name). None if no row qualifies."""
    # TODO 1: filter, then min(..., key=(latency, -accuracy, name))
    raise NotImplementedError("TODO 1: choose")


def markdown_table(rows: list[dict]) -> str:
    """'| model | accuracy | latency (s) | errors |' header, separator, then rows sorted by accuracy descending (ties: name).
    accuracy with 2 decimals; latency with 2 decimals or '-' when None."""
    # TODO 2: build the lines and join them with "\n"
    raise NotImplementedError("TODO 2: markdown_table")


if __name__ == "__main__":
    rows = [{"model": "ollama:qwen2.5:7b", "accuracy": 0.75, "mean_latency_s": 1.2, "errors": 0, "n": 4},
            {"model": "anthropic:claude", "accuracy": 1.0, "mean_latency_s": 0.9, "errors": 0, "n": 4},
            {"model": "opencode:x/y", "accuracy": 1.0, "mean_latency_s": 3.0, "errors": 0, "n": 4}]
    print(markdown_table(rows))
    print("chosen:", choose(rows))


In [ ]:
from learn.notebook import check_cell
check_cell('m01_compare_models', 'step03_choose_and_report', globals())